# Evaluation Suite -- Seq2Seq Chatbot Models

Full evaluation pipeline for **Baseline** and **Attention** Seq2Seq models.

**Outputs produced per run:**
- `bleu_results.json` -- BLEU-1/2/3/4 (sacrebleu 13a) + ROUGE-L + Distinct-1/2 + BERTScore
- `baseline_manual_samples.json` / `attention_manual_samples.json`
- `attention_heatmap.png` -- Bahdanau attention weight heatmap (attention model only)

**Benchmark reference** (osamadev/seq2seq-chatbot -- LSTM+ATTN, greedy, 8k SP vocab):

| Metric | Reference |
|--------|-----------|
| BLEU-1 | 0.4400 |
| BLEU-4 | 0.1386 |
| ROUGE-L F1 | 0.0922 |

> **NOTE (F3):** Direct numeric comparison is invalid -- different tokeniser (8k SentencePiece vs our 16k), different vocab size, and different data split. Use as directional reference only.

**Metric rationale** (G1 / Liu et al. 2016): BLEU alone is insufficient for open-domain dialogue. We additionally report Distinct-1/2 (response diversity), BERTScore F1 (semantic similarity), and ROUGE-L (longest-common-subsequence recall).

In [ ]:
import json
import os
import math
import random
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn.functional as F
import sentencepiece as spm
import sacrebleu
from rouge_score import rouge_scorer
import bert_score
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is on sys.path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"torch  : {torch.__version__}")
print(f"device : {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Decoding Functions

Two decoding strategies:
- **Greedy decode** -- deterministic argmax at each step; used for BLEU corpus evaluation (matches osamadev benchmark methodology).
- **Top-p (nucleus) decode** -- stochastic sampling with temperature scaling and n-gram blocking (A4); used for manual / qualitative evaluation.

In [ ]:
@torch.inference_mode()
def greedy_decode(
    model,
    src: torch.Tensor,          # [batch, src_len]
    src_lengths: torch.Tensor,  # [batch]
    sos_idx: int,
    eos_idx: int,
    max_len: int,
    device: torch.device,
) -> List[List[int]]:
    """
    Greedy decode a batch. Returns list of token ID lists (no padding, no EOS).
    Used for BLEU computation -- matches osamadev benchmark methodology (F3).
    """
    model.eval()
    src = src.to(device)
    src_lengths = src_lengths.to(device)
    batch_size = src.size(0)

    # Encode once.
    encoder_outputs, (h_n, c_n) = model.encoder(src, src_lengths)
    src_mask = (src == model.encoder.embedding.padding_idx)  # [B, src_len]
    dec_h, dec_c = model.bridge(h_n, c_n)

    input_token = torch.full(
        (batch_size,), sos_idx, dtype=torch.long, device=device
    )

    decoded: List[List[int]] = [[] for _ in range(batch_size)]
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    # Initial context vector (zeros); updated each step by decoder.
    context = torch.zeros(batch_size, encoder_outputs.size(-1), device=device)

    for _ in range(max_len):
        logits, dec_h, dec_c, context, _ = model.decoder.forward_step(
            input_token, dec_h, dec_c, encoder_outputs, context, src_mask
        )
        next_token = logits.argmax(dim=-1)  # [batch]

        for i in range(batch_size):
            if not finished[i]:
                tok = next_token[i].item()
                if tok == eos_idx:
                    finished[i] = True
                else:
                    decoded[i].append(tok)

        if finished.all():
            break

        input_token = next_token

    return decoded


@torch.inference_mode()
def top_p_decode(
    model,
    src: torch.Tensor,          # [batch, src_len]
    src_lengths: torch.Tensor,  # [batch]
    sos_idx: int,
    eos_idx: int,
    max_len: int,
    device: torch.device,
    top_p: float = 0.9,
    temperature: float = 0.8,
    ngram_block: int = 3,
) -> List[List[int]]:
    """
    Top-p (nucleus) sampling with temperature scaling and n-gram blocking (A4).
    Used for interactive / qualitative evaluation; not used for BLEU benchmark.
    """
    model.eval()
    src = src.to(device)
    src_lengths = src_lengths.to(device)
    batch_size = src.size(0)

    encoder_outputs, (h_n, c_n) = model.encoder(src, src_lengths)
    src_mask = (src == model.encoder.embedding.padding_idx)
    dec_h, dec_c = model.bridge(h_n, c_n)

    input_token = torch.full(
        (batch_size,), sos_idx, dtype=torch.long, device=device
    )

    decoded: List[List[int]] = [[] for _ in range(batch_size)]
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
    context = torch.zeros(batch_size, encoder_outputs.size(-1), device=device)

    for _ in range(max_len):
        logits, dec_h, dec_c, context, _ = model.decoder.forward_step(
            input_token, dec_h, dec_c, encoder_outputs, context, src_mask
        )

        next_tokens = []
        for i in range(batch_size):
            if finished[i]:
                next_tokens.append(input_token[i])
                continue

            lgts = logits[i].clone()

            # N-gram blocking (A4): pre-softmax logit masking
            if ngram_block > 1 and len(decoded[i]) >= ngram_block - 1:
                prefix = tuple(decoded[i][-(ngram_block - 1):])
                blocked = set()
                seq = decoded[i]
                for start in range(len(seq) - (ngram_block - 1)):
                    if tuple(seq[start: start + ngram_block - 1]) == prefix:
                        blocked.add(seq[start + ngram_block - 1])
                if blocked and len(blocked) < lgts.size(0):
                    lgts[list(blocked)] = float("-inf")

            # Temperature scaling
            lgts = lgts / max(temperature, 1e-8)

            # Nucleus (top-p) filtering
            probs = F.softmax(lgts, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cumulative = sorted_probs.cumsum(dim=-1)
            remove_mask = cumulative - sorted_probs > top_p
            sorted_probs[remove_mask] = 0.0
            sorted_probs /= sorted_probs.sum().clamp(min=1e-8)

            sampled_pos = torch.multinomial(sorted_probs, num_samples=1)
            next_tok = sorted_idx[sampled_pos].item()

            if next_tok == eos_idx:
                finished[i] = True
                next_tokens.append(input_token[i])
            else:
                decoded[i].append(next_tok)
                next_tokens.append(torch.tensor(next_tok, device=device))

        input_token = torch.stack(next_tokens)

        if finished.all():
            break

    return decoded

print("Decoding functions defined: greedy_decode, top_p_decode")

## Metric Helpers

- **`compute_distinct_n`** -- Corpus-level Distinct-N: unique n-grams / total n-grams (R5 / G2). Measures response diversity.
- **`_ids_to_str`** -- Decode BPE token IDs back to a detokenised string via SentencePiece.

In [ ]:
def compute_distinct_n(sequences: List[List[int]], n: int) -> float:
    """
    Corpus-level Distinct-N: unique n-grams / total n-grams (R5 / G2).
    Returns 0.0 if there are no n-grams (degenerate empty output).
    """
    total = 0
    unique: set = set()
    for seq in sequences:
        for i in range(len(seq) - n + 1):
            gram = tuple(seq[i: i + n])
            unique.add(gram)
            total += 1
    return len(unique) / max(total, 1)


def _ids_to_str(ids: List[int], sp: spm.SentencePieceProcessor) -> str:
    """Decode BPE IDs to a detokenised string, skipping special tokens."""
    return sp.decode(ids)

print("Metric helpers defined: compute_distinct_n, _ids_to_str")

## Corpus-Level Metrics: BLEU + ROUGE + Distinct + BERTScore

Computes all corpus-level metrics using **greedy decoding** (fair comparison with osamadev benchmark -- F3):
- **BLEU-1/2/3/4** via sacrebleu with 13a tokeniser. Each BLEU-N is an independent `corpus_bleu` call with its own brevity penalty (AC2-I2).
- **ROUGE-L** F1 -- longest-common-subsequence recall.
- **Distinct-1/2** -- response diversity (G2 / R5).
- **BERTScore F1** -- semantic similarity beyond n-gram overlap (G1).

In [ ]:
def compute_bleu_corpus(
    model,
    loader,
    sp: spm.SentencePieceProcessor,
    device: torch.device,
    max_len: int = 40,
    sos_idx: int = 2,
    eos_idx: int = 3,
    bert_score_model: str = "distilbert-base-uncased",
    bert_score_batch: int = 64,
) -> Dict[str, float]:
    """
    Corpus-level evaluation: BLEU-1/2/3/4 (sacrebleu 13a), ROUGE-L, Distinct-1/2,
    BERTScore F1.  Uses greedy decoding for benchmark comparability (F3).
    """
    hypotheses_str: List[str] = []
    references_str: List[str] = []
    hypotheses_ids: List[List[int]] = []

    for batch in loader:
        src = batch["src"].to(device)
        src_lengths = batch["src_lengths"]
        trg = batch["trg"]  # [B, trg_len] -- reference (CPU)

        hyp_ids = greedy_decode(
            model, src, src_lengths,
            sos_idx=sos_idx, eos_idx=eos_idx,
            max_len=max_len, device=device,
        )

        for i in range(src.size(0)):
            # Reference: strip sos/eos tokens.
            ref_ids = trg[i].tolist()
            if ref_ids and ref_ids[0] == sos_idx:
                ref_ids = ref_ids[1:]
            ref_ids = [t for t in ref_ids if t not in (eos_idx, 0)]  # strip eos+pad

            hypotheses_str.append(_ids_to_str(hyp_ids[i], sp))
            references_str.append(_ids_to_str(ref_ids, sp))
            hypotheses_ids.append(hyp_ids[i])

    # BLEU (sacrebleu 13a)
    bleu = sacrebleu.corpus_bleu(hypotheses_str, [references_str], tokenize="13a")
    bleu1 = sacrebleu.corpus_bleu(hypotheses_str, [references_str], tokenize="13a", max_ngram_order=1).score / 100
    bleu2 = sacrebleu.corpus_bleu(hypotheses_str, [references_str], tokenize="13a", max_ngram_order=2).score / 100
    bleu3 = sacrebleu.corpus_bleu(hypotheses_str, [references_str], tokenize="13a", max_ngram_order=3).score / 100
    bleu4 = bleu.score / 100

    # ROUGE-L
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    rougeL_scores = [
        scorer.score(ref, hyp)["rougeL"].fmeasure
        for ref, hyp in zip(references_str, hypotheses_str)
    ]
    rougeL_f1 = sum(rougeL_scores) / max(len(rougeL_scores), 1)

    # Distinct-1/2
    distinct1 = compute_distinct_n(hypotheses_ids, 1)
    distinct2 = compute_distinct_n(hypotheses_ids, 2)

    # BERTScore
    P, R, F1 = bert_score.score(
        hypotheses_str,
        references_str,
        lang="en",
        model_type=bert_score_model,
        batch_size=bert_score_batch,
        device=str(device),
        verbose=False,
    )
    bertscore_f1 = F1.mean().item()

    results = {
        "bleu1": round(bleu1, 4),
        "bleu2": round(bleu2, 4),
        "bleu3": round(bleu3, 4),
        "bleu4": round(bleu4, 4),
        "rougeL_f1": round(rougeL_f1, 4),
        "distinct1": round(distinct1, 4),
        "distinct2": round(distinct2, 4),
        "bertscore_f1": round(bertscore_f1, 4),
        "num_samples": len(hypotheses_str),
        "decode_strategy": "greedy",
    }

    return results

print("compute_bleu_corpus defined")

## Manual Evaluation Samples

Generate `{src, tgt, hyp}` dicts for qualitative inspection. Uses **top-p sampling** by default for response diversity, with n-gram blocking to prevent repetition.

In [ ]:
def manual_evaluation_samples(
    model,
    loader,
    sp: spm.SentencePieceProcessor,
    device: torch.device,
    num_samples: int = 50,
    decode_strategy: str = "top_p",
    sos_idx: int = 2,
    eos_idx: int = 3,
    max_len: int = 40,
    top_p: float = 0.9,
    temperature: float = 0.8,
    ngram_block: int = 3,
) -> List[Dict[str, str]]:
    """
    Generate num_samples {src, tgt, hyp} dicts for manual inspection.
    decode_strategy: "greedy" or "top_p" (default top_p for diversity).
    """
    samples: List[Dict[str, str]] = []

    for batch in loader:
        if len(samples) >= num_samples:
            break

        src = batch["src"].to(device)
        src_lengths = batch["src_lengths"]
        trg = batch["trg"]

        if decode_strategy == "greedy":
            hyp_ids = greedy_decode(
                model, src, src_lengths,
                sos_idx=sos_idx, eos_idx=eos_idx,
                max_len=max_len, device=device,
            )
        else:
            hyp_ids = top_p_decode(
                model, src, src_lengths,
                sos_idx=sos_idx, eos_idx=eos_idx,
                max_len=max_len, device=device,
                top_p=top_p, temperature=temperature, ngram_block=ngram_block,
            )

        for i in range(src.size(0)):
            if len(samples) >= num_samples:
                break

            src_ids = src[i].tolist()
            src_ids = [t for t in src_ids if t not in (0,)]  # strip pad
            ref_ids = trg[i].tolist()
            if ref_ids and ref_ids[0] == sos_idx:
                ref_ids = ref_ids[1:]
            ref_ids = [t for t in ref_ids if t not in (eos_idx, 0)]

            samples.append({
                "src": _ids_to_str(src_ids, sp),
                "tgt": _ids_to_str(ref_ids, sp),
                "hyp": _ids_to_str(hyp_ids[i], sp),
            })

    return samples

print("manual_evaluation_samples defined")

## Attention Heatmap Visualisation (G4)

Decode a single source sequence with the attention model, collect per-step Bahdanau attention weights, and render a heatmap. Only works with the attention model (BaselineDecoder has no attention mechanism).

In [ ]:
@torch.inference_mode()
def plot_attention_heatmap(
    model,
    src_ids: List[int],
    sp: spm.SentencePieceProcessor,
    device: torch.device,
    save_path: Optional[str] = None,
    sos_idx: int = 2,
    eos_idx: int = 3,
    max_len: int = 40,
) -> None:
    """
    Decode a single src sequence, collect per-step attention weights,
    and display/save a heatmap (G4).

    Works only with the attention model (BaselineDecoder has no attention weights).
    If the model has no attention mechanism, this is a no-op.
    """
    if not hasattr(model.decoder, "attention"):
        print("Model has no attention mechanism -- skipping heatmap.")
        return

    model.eval()
    src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)
    src_lengths = torch.tensor([len(src_ids)], dtype=torch.long)

    encoder_outputs, (h_n, c_n) = model.encoder(src_tensor, src_lengths)
    src_mask = (src_tensor == model.encoder.embedding.padding_idx)
    dec_h, dec_c = model.bridge(h_n, c_n)

    input_token = torch.tensor([sos_idx], dtype=torch.long, device=device)
    decoded_ids: List[int] = []
    attn_weights: List[torch.Tensor] = []  # each: [src_len]

    context = torch.zeros(1, encoder_outputs.size(-1), device=device)

    for _ in range(max_len):
        logits, dec_h, dec_c, context, step_attn = model.decoder.forward_step(
            input_token, dec_h, dec_c, encoder_outputs, context, src_mask
        )
        next_tok = logits.argmax(dim=-1).item()
        if next_tok == eos_idx:
            break
        decoded_ids.append(next_tok)
        if step_attn is not None:
            attn_weights.append(step_attn.squeeze(0).cpu())  # [src_len]
        input_token = torch.tensor([next_tok], dtype=torch.long, device=device)

    if not attn_weights:
        print("No attention weights collected -- nothing to plot.")
        return

    # Build heatmap matrix: [trg_len, src_len]
    attn_matrix = torch.stack(attn_weights, dim=0).numpy()

    src_pieces = sp.encode(sp.decode(src_ids), out_type=str) if src_ids else ["?"]
    trg_pieces = sp.encode(sp.decode(decoded_ids), out_type=str) if decoded_ids else ["?"]

    fig_w = max(8, len(src_pieces) * 0.4)
    fig_h = max(4, len(trg_pieces) * 0.35)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    sns.heatmap(
        attn_matrix,
        xticklabels=src_pieces,
        yticklabels=trg_pieces,
        ax=ax,
        cmap="YlOrRd",
        linewidths=0.3,
        cbar_kws={"label": "Attention weight"},
    )
    ax.set_xlabel("Source tokens")
    ax.set_ylabel("Generated tokens")
    ax.set_title("Bahdanau Attention Weights")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f"Heatmap saved to {save_path}")

    plt.show()
    plt.close(fig)

print("plot_attention_heatmap defined")

## Setup

Load configuration, set seed for reproducibility, initialise device, load SentencePiece model, and build the test dataloader.

In [ ]:
from config import CONFIG, set_seed
from dataset import build_dataloaders
from models import build_model

# Reproducibility
set_seed(CONFIG.get("seed", 42))

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Paths
artifact_dir = Path(CONFIG["artifact_dir"])
checkpoint_dir = Path(CONFIG["checkpoint_dir"])
print(f"Artifact dir   : {artifact_dir}")
print(f"Checkpoint dir : {checkpoint_dir}")

# Load SentencePiece processor
sp = spm.SentencePieceProcessor(model_file=str(artifact_dir / "stage5_spm.model"))
print(f"SPM vocab size : {sp.get_piece_size()}")

# Build test dataloader
_, _, test_loader = build_dataloaders(
    artifact_dir=str(artifact_dir),
    batch_size=CONFIG.get("batch_size", 128),
    num_workers=0,
    max_ctx_len=CONFIG.get("max_ctx_tokens", 100),
    max_resp_len=CONFIG.get("max_resp_tokens", 40) + 2,
    pad_idx=CONFIG.get("pad_idx", 0),
)
print(f"Test batches   : {len(test_loader)}")

# Common indices
sos_idx = CONFIG.get("sos_idx", 2)
eos_idx = CONFIG.get("eos_idx", 3)
max_len = CONFIG.get("max_decode_len", 40)

## Run Evaluation

Load best checkpoints for both models, compute all corpus metrics, generate manual samples, and produce the attention heatmap. Results are saved to the checkpoint directory.

In [ ]:
all_bleu: Dict[str, Dict] = {}

for model_type in ("baseline", "attention"):
    ckpt_path = checkpoint_dir / f"{model_type}_best.pt"
    if not ckpt_path.exists():
        print(f"[evaluate] {ckpt_path} not found -- skipping {model_type}")
        continue

    print(f"\n{'='*60}")
    print(f"  {model_type.upper()} MODEL")
    print(f"{'='*60}")

    ckpt = torch.load(str(ckpt_path), map_location=device, weights_only=False)
    model = build_model(model_type, CONFIG, device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    # -- Corpus metrics (BLEU / ROUGE / Distinct / BERTScore) --
    print("  Computing corpus metrics...")
    metrics = compute_bleu_corpus(
        model, test_loader, sp, device,
        max_len=max_len, sos_idx=sos_idx, eos_idx=eos_idx,
    )
    all_bleu[model_type] = metrics
    for k, v in metrics.items():
        print(f"    {k}: {v}")

    # -- Manual samples --
    print("  Generating manual evaluation samples...")
    samples = manual_evaluation_samples(
        model, test_loader, sp, device,
        num_samples=50, decode_strategy="top_p",
        sos_idx=sos_idx, eos_idx=eos_idx, max_len=max_len,
        top_p=CONFIG.get("top_p", 0.9),
        temperature=CONFIG.get("temperature", 0.8),
        ngram_block=CONFIG.get("ngram_block", 3),
    )
    samples_path = checkpoint_dir / f"{model_type}_manual_samples.json"
    with open(str(samples_path), "w", encoding="utf-8") as f:
        json.dump(samples, f, indent=2, ensure_ascii=False)
    print(f"  Saved {len(samples)} samples -> {samples_path}")

    # -- Attention heatmap (attention model only) --
    if model_type == "attention":
        print("  Generating attention heatmap...")
        sample_batch = next(iter(test_loader))
        src_ids = [t for t in sample_batch["src"][0].tolist() if t != 0]
        heatmap_path = str(checkpoint_dir / "attention_heatmap.png")
        plot_attention_heatmap(
            model, src_ids, sp, device, save_path=heatmap_path,
            sos_idx=sos_idx, eos_idx=eos_idx, max_len=max_len,
        )

# Save combined results (atomic write)
bleu_path = checkpoint_dir / "bleu_results.json"
_tmp_bleu = str(bleu_path) + ".tmp"
with open(_tmp_bleu, "w") as f:
    json.dump(all_bleu, f, indent=2)
os.replace(_tmp_bleu, str(bleu_path))
print(f"\nAll metrics saved -> {bleu_path}")

## Results -- Comparison Table

Side-by-side comparison of Baseline vs Attention model metrics, with the osamadev directional reference (F3).

> **\* Reference:** osamadev/seq2seq-chatbot (8k SP, greedy) -- directional only; different tokeniser and vocab size.

In [ ]:
# Display comparison table
ref = {"bleu1": 0.4400, "bleu4": 0.1386, "rougeL_f1": 0.0922}
metrics_order = ("bleu1", "bleu2", "bleu3", "bleu4", "rougeL_f1", "distinct1", "distinct2", "bertscore_f1")

header = f"{'Metric':<20} {'Baseline':>12} {'Attention':>12} {'Reference*':>12}"
sep = "-" * 70

print("=" * 70)
print(header)
print(sep)
for metric in metrics_order:
    b_val = all_bleu.get("baseline", {}).get(metric, "---")
    a_val = all_bleu.get("attention", {}).get(metric, "---")
    r_val = ref.get(metric, "---")
    print(f"  {metric:<18} {str(b_val):>12} {str(a_val):>12} {str(r_val):>12}")
print("=" * 70)
print("* Reference: osamadev/seq2seq-chatbot (8k SP, greedy) -- directional only (F3)")

# Also display as a dict for programmatic access
print("\nRaw results dict:")
print(json.dumps(all_bleu, indent=2))